In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/SYNESIS

/content/drive/MyDrive/SYNESIS


In [ ]:
%cd /content/drive/MyDrive/SYNESIS/synesis

/content/drive/MyDrive/SYNESIS/synesis


In [ ]:
#!git clone -b main https://github.com/03gonzaleznerea/synesis.git

fatal: destination path 'synesis' already exists and is not an empty directory.


In [ ]:
!ls


assets	externals  NOTICE     requirements.txt	synesis
config	LICENSE    README.md  ruff.toml


In [ ]:
!pip install -r requirements.txt

In [ ]:
!ls

assets	externals  NOTICE     requirements.txt	synesis
config	LICENSE    README.md  ruff.toml


In [ ]:
%cd config

/content/drive/MyDrive/SYNESIS/synesis/config


In [ ]:
%cd /content/drive/MyDrive/SYNESIS/synesis

/content/drive/MyDrive/SYNESIS/synesis


In [ ]:
!ls synesis/datasets

alltempo.py	  imagenet.py		     librispeech.py    mtgjamendo.py
dataset_utils.py  __init__.py		     magnatagatune.py  syntheory.py
giantstepskey.py  librispeech_durations.csv  mpe.py	       tinysol.py


CONFIGURATION OF THE DATASET


In [ ]:
!mkdir -p /content/drive/MyDrive/datasets/nsynth

In [ ]:
import os, zipfile, shutil, random
from pathlib import Path

# Ruta donde tienes los datos en Drive
DATA_ROOT = Path("/content/drive/MyDrive/TFG_AFTER/data")

# Carpeta final del dataset
OUT_ROOT = Path("/content/drive/MyDrive/datasets/nsynth")

OUT_ROOT.mkdir(parents=True, exist_ok=True)

print(os.listdir(DATA_ROOT))

['nsynth-test.jsonwav.tar.gz', 'nsynth-test', 'nsynth-valid.jsonwav.tar.gz']


In [ ]:
for zip_path in DATA_ROOT.glob("*.zip"):
    extract_folder = DATA_ROOT / zip_path.stem

    if extract_folder.exists():
        print(f"Ya existe: {extract_folder}")
    else:
        print(f"Extrayendo {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(DATA_ROOT)

print("Listo")

Listo


In [ ]:
wav_files = list(DATA_ROOT.rglob("*.wav"))

print("Total wavs encontrados:", len(wav_files))
print(wav_files[:5])

Total wavs encontrados: 4096
[PosixPath('/content/drive/MyDrive/TFG_AFTER/data/nsynth-test/audio/guitar_electronic_022-081-050.wav'), PosixPath('/content/drive/MyDrive/TFG_AFTER/data/nsynth-test/audio/bass_electronic_027-056-050.wav'), PosixPath('/content/drive/MyDrive/TFG_AFTER/data/nsynth-test/audio/mallet_acoustic_062-092-100.wav'), PosixPath('/content/drive/MyDrive/TFG_AFTER/data/nsynth-test/audio/bass_synthetic_009-093-075.wav'), PosixPath('/content/drive/MyDrive/TFG_AFTER/data/nsynth-test/audio/guitar_acoustic_010-022-075.wav')]


In [ ]:
random.seed(42)
random.shuffle(wav_files)

n = len(wav_files)

n_train = int(0.8 * n)
n_val = int(0.1 * n)

train_files = wav_files[:n_train]
val_files = wav_files[n_train:n_train+n_val]
test_files = wav_files[n_train+n_val:]

splits = {
    "train": train_files,
    "val": val_files,
    "test": test_files
}

for split_name, files in splits.items():
    split_dir = OUT_ROOT / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    for f in files:
        dest = split_dir / f.name
        shutil.copy2(f, dest)

print("Dataset creado:")
print("Train:", len(train_files))
print("Val:", len(val_files))
print("Test:", len(test_files))

Dataset creado:
Train: 3276
Val: 409
Test: 411


In [ ]:
import os

os.rename(
"/content/drive/MyDrive/datasets/nsynth/val",
"/content/drive/MyDrive/datasets/nsynth/validation"
)

In [ ]:
!find . -maxdepth 3 -type d | grep datasets

./synesis/datasets


In [ ]:
!find . -maxdepth 3 -type f | grep imagenet

./synesis/datasets/imagenet.py
./synesis/features/resnet101_imagenet.py
./synesis/features/resnet18_imagenet.py
./synesis/features/resnet34_imagenet.py
./synesis/features/resnet50_imagenet.py
./synesis/features/vit_b_16_imagenet.py
./synesis/features/vit_b_32_imagenet.py
./synesis/features/vit_l_16_imagenet.py
./synesis/features/vit_l_32_imagenet.py


In [ ]:
!touch synesis/datasets/nsynth.py
!ls synesis/datasets

alltempo.py	  __init__.py		     mpe.py	    tinysol.py
dataset_utils.py  librispeech_durations.csv  mtgjamendo.py
giantstepskey.py  librispeech.py	     nsynth.py
imagenet.py	  magnatagatune.py	     syntheory.py


In [ ]:
!sed -n '1,220p' synesis/datasets/tinysol.py

from pathlib import Path
from typing import Optional, Tuple, Union

import mirdata
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import Tensor
from torch.utils.data import Dataset

from config.features import configs as feature_configs
from synesis.datasets.dataset_utils import load_track


def pitch_to_midi(pitch_str: str) -> int:
    """Convert pitch string to MIDI note number."""
    # Define pitch classes
    pitch_classes = {
        "C": 0,
        "C#": 1,
        "Db": 1,
        "D": 2,
        "D#": 3,
        "Eb": 3,
        "E": 4,
        "F": 5,
        "F#": 6,
        "Gb": 6,
        "G": 7,
        "G#": 8,
        "Ab": 8,
        "A": 9,
        "A#": 10,
        "Bb": 10,
        "B": 11,
    }

    # Handle special cases for sharp/flat notes
    if len(pitch_str) == 3:
        pitch_class = pitch_str[:2]  # e.g., 'A#' from 'A#3'
        octave = int(pitch_str[2])
    else:
        pitch

In [ ]:
%%writefile synesis/datasets/nsynth.py
from pathlib import Path
from typing import Optional, Tuple, Union
import json

import torch
from sklearn.preprocessing import LabelEncoder
from torch import Tensor
from torch.utils.data import Dataset

from config.features import configs as feature_configs
from synesis.datasets.dataset_utils import load_track


class Nsynth(Dataset):
    def __init__(
        self,
        feature: str,
        root: Union[str, Path] = "/content/drive/MyDrive/datasets/nsynth",
        split: Optional[str] = None,
        download: bool = False,
        feature_config: Optional[dict] = None,
        audio_format: str = "wav",
        item_format: str = "feature",
        itemization: bool = True,
        fv: str = "pitch",
        seed: int = 42,
    ) -> None:
        self.tasks = [
            "pitch_classification",
            "instrument_family_classification",
            "instrument_source_classification",
        ]
        self.fvs = ["pitch", "instrument_family", "instrument_source", "instrument"]
        assert fv in self.fvs, f"Invalid factor of variation: {fv}"
        self.fv = fv

        self.root = Path(root)
        if split not in [None, "train", "test", "validation"]:
            raise ValueError(
                f"Invalid split: {split}. Options: None, 'train', 'test', 'validation'"
            )

        self.split = split
        self.item_format = item_format
        self.itemization = itemization
        self.audio_format = audio_format
        self.feature = feature
        self.seed = seed

        if download:
            raise NotImplementedError("Download not supported for NSynth")

        if not feature_config:
            feature_config = feature_configs[feature]
        self.feature_config = feature_config

        self.label_encoder = LabelEncoder()
        self._load_metadata()

    def _load_metadata(self) -> Tuple[list, torch.Tensor]:
        splits = ["train", "validation", "test"] if self.split is None else [self.split]

        paths = []
        labels = []

        for split in splits:
            split_dir = self.root / split

            audio_dir = split_dir / "audio"
            if not audio_dir.exists():
                audio_dir = split_dir

            json_path = split_dir / "examples.json"

            metadata = {}
            if json_path.exists():
                with open(json_path, "r") as f:
                    metadata = json.load(f)

            wav_paths = sorted(audio_dir.glob(f"*.{self.audio_format}"))

            for wav_path in wav_paths:
                paths.append(str(wav_path))
                note_id = wav_path.stem

                if metadata and note_id in metadata:
                    item = metadata[note_id]

                    if self.fv == "pitch":
                        labels.append(item["pitch"])
                    elif self.fv == "instrument_family":
                        labels.append(item["instrument_family_str"])
                    elif self.fv == "instrument_source":
                        labels.append(item["instrument_source_str"])
                    elif self.fv == "instrument":
                        labels.append(item["instrument_str"])
                else:
                    # fallback si no tienes examples.json
                    labels.append(0)

        labels = self.label_encoder.fit_transform(labels)
        labels = torch.tensor(labels, dtype=torch.long)

        self.feature_paths = [
            path.replace(f".{self.audio_format}", ".pt")
            .replace("/audio/", f"/{self.feature}/")
            for path in paths
        ]

        self.raw_data_paths = paths
        self.labels = labels
        self.paths = (
            self.raw_data_paths if self.item_format == "raw" else self.feature_paths
        )

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> Tuple[Tensor, Tensor]:
        path = (
            self.raw_data_paths[idx]
            if self.item_format == "raw"
            else self.feature_paths[idx]
        )

        label = self.labels[idx]

        track = load_track(
            path=path,
            item_format=self.item_format,
            itemization=self.itemization,
            item_len_sec=self.feature_config["item_len_sec"],
            sample_rate=self.feature_config["sample_rate"],
        )

        return track, label

Overwriting synesis/datasets/nsynth.py


In [ ]:
from synesis.datasets.nsynth import Nsynth

ds = Nsynth(
    feature="after",
    root="/content/drive/MyDrive/datasets/nsynth",
    split="train",
    item_format="raw",
    fv="pitch",
)

print(len(ds))
x, y = ds[0]
print(type(x), y)

KeyError: 'after'

In [ ]:
from config.features import configs as feature_configs
print(feature_configs.keys())

dict_keys(['MDuo', 'MelSpec', 'AudioMAE', 'VGGishMTAT', 'MULE_1728', 'MULE_512', 'Music2Latent_8192', 'Music2Latent_64', 'PESTO', 'MERT', 'Wav2Vec2', 'HuBERT', 'CLAP', 'Whisper', 'UniSpeech', 'XVector', 'ResNet18_ImageNet', 'ResNet34_ImageNet', 'ResNet50_ImageNet', 'ResNet101_ImageNet', 'ViT_ImageNet', 'ViT_b_16_ImageNet', 'ViT_l_16_ImageNet', 'ViT_b_32_ImageNet', 'ViT_l_32_ImageNet', 'SimCLR', 'DINO', 'DINOv2', 'DINOv2_small', 'DINOv2_base', 'DINOv2_large', 'ViT_MAE', 'CLIP', 'IJEPA', 'AFTER_Timbre', 'AFTER_Structure', 'AFTER_Combined', 'SSVQVAE_Combined', 'SSVQVAE_Structure', 'SSVQVAE_Timbre', 'TSDSAE_Structure', 'TSDSAE_Timbre', 'TSDSAE_Combined', 'AFTER_Timbre_no_adv', 'AFTER_Structure_no_adv', 'AFTER_Timbre_no_augm', 'AFTER_Structure_no_augm'])


In [ ]:
from synesis.datasets.nsynth import Nsynth

ds = Nsynth(
    feature="AFTER_Timbre_no_adv",
    root="/content/drive/MyDrive/datasets/nsynth",
    split="train",
    item_format="raw",
    fv="pitch"
)

print(len(ds))

x, y = ds[0]
print(type(x), y)

3276


KeyError: 'item_len_sec'

In [ ]:
from config.features import configs as feature_configs

feature_config = feature_configs["AFTER_Timbre_no_adv"]
print(feature_config)

{'__cls__': 'AFTER_Timbre', 'sample_rate': 44100, 'extract_kws': {'autoencoder_path': './externals/AFTER/pretrained/AE_slakh.pt', 'checkpoint_path': './externals/AFTER/experiments/after_runs/slakh2100_train_no_adv/checkpoint1000000_EMA.pt', 'config_path': './externals/AFTER/experiments/after_runs/slakh2100_train_no_adv/config.gin'}}


In [ ]:
ds = Nsynth(
    feature="AFTER_Timbre_no_adv",
    root="/content/drive/MyDrive/datasets/nsynth",
    split="train",
    item_format="raw",
    fv="pitch",
    feature_config={
        **feature_configs["AFTER_Timbre_no_adv"],
        "item_len_sec": 4,
        "sample_rate": 16000,
    }
)

print(len(ds))
x, y = ds[0]
print(type(x), y)

3276
<class 'torch.Tensor'> tensor(0)


In [ ]:
from config.features import configs as feature_configs

ds = Nsynth(
    feature="AFTER_Timbre_no_adv",
    root="/content/drive/MyDrive/datasets/nsynth",
    split="train",
    item_format="raw",
    fv="pitch",
    feature_config={
        **feature_configs["AFTER_Timbre_no_adv"],
        "item_len_sec": 4,
        "sample_rate": 16000
    }
)

print(len(ds))
x,y = ds[0]
print(type(x), y)

3276
<class 'torch.Tensor'> tensor(0)


In [ ]:
print(x.shape)

torch.Size([2, 1, 64000])


In [ ]:
print(x[0].shape)
print(x[1].shape)

torch.Size([1, 64000])
torch.Size([1, 64000])


In [ ]:
fv="instrument_family"

In [ ]:
!python -m synesis.extract -f AFTER_Timbre_no_adv -d nsynth -b 4

Traceback (most recent call last):
  File "/content/drive/MyDrive/SYNESIS/synesis/synesis/features/feature_utils.py", line 37, in get_feature_extractor
    module = importlib.import_module(f"synesis.features.{__cls__.lower()}")
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/content/drive/MyDrive/SYNESIS/synesis/synesis/f